# Food Map — Import Pipeline

**Steps:**
1. Fetch OpenAlex counts for all ~50 foods (determines node sizes in the map)
2. Browse the available foods and pick which ones to import
3. Run the import (fetches papers from OpenAlex into SQLite)
4. Build the frontend JSON files (`food_universe.json` + per-food `nodes/edges.json`)
5. Inspect what was imported

Each node in the Food Map represents a specific food (broccoli, salmon, oats, etc.).
Papers are those that mention the food in the context of infant/child nutrition.

In [1]:
import sys, os, glob, sqlite3

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.abspath(os.path.join(BACKEND_DIR, "..", "data"))
OUT_DIR     = os.path.abspath(os.path.join(BACKEND_DIR, "..", "frontend", "public"))

sys.path.insert(0, BACKEND_DIR)
from import_foods import PREDEFINED_FOODS, import_food, fetch_all_food_counts
from build_food_data import build_food_universe

print(f"Backend : {BACKEND_DIR}")
print(f"Data    : {DATA_DIR}")
print(f"Output  : {OUT_DIR}")
print(f"Foods available: {len(PREDEFINED_FOODS)}")

Backend : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\backend
Data    : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data
Output  : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\frontend\public
Foods available: 98


## Step 1 — Fetch OpenAlex paper counts

Makes one lightweight API call per food (no paper download) and saves counts to
`data/food_counts.json`. These totals drive node size in the Food Map.

Run this once upfront; re-run anytime to refresh the counts.

In [4]:
counts = fetch_all_food_counts(out_path=os.path.join(DATA_DIR, "food_counts.json"))
print(f"\nFetched counts for {len(counts)} foods.")
top10 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 by total OpenAlex papers:")
for k, v in top10:
    cfg = PREDEFINED_FOODS.get(k, {})
    print(f"  {cfg.get('name', k):25s}  {v:>8,d}")

  [  1/98] broccoli                                  18
  [  2/98] carrot                                    23
  [  3/98] sweet_potato                              35
  [  4/98] spinach                                    1
  [  5/98] pea                                       68
  [  6/98] avocado                                   13
  [  7/98] tomato                                    37
  [  8/98] zucchini                                   2
  [  9/98] cauliflower                               12
  [ 10/98] apple                                     14
  [ 11/98] banana                                    15
  [ 12/98] mango                                     67
  [ 13/98] strawberry                                13
  [ 14/98] blueberry                                 18
  [ 15/98] pear                                       5
  [ 16/98] orange                                    58
  [ 17/98] grape                                      0
  [ 18/98] watermelon                           

## Step 2 — Browse available foods

All foods grouped by category. Foods that already have a local database show their current paper count.

In [2]:
def db_paper_count(food_key):
    path = os.path.join(DATA_DIR, f'food_papers_{food_key}.db')
    if not os.path.exists(path):
        return None
    try:
        conn = sqlite3.connect(path)
        n = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

from import_foods import FOOD_GROUP_ORDER

# Group foods by their category
by_group = {g: [] for g in FOOD_GROUP_ORDER}
for key, cfg in PREDEFINED_FOODS.items():
    by_group.setdefault(cfg['group'], []).append((key, cfg))

total_imported = 0
for group in FOOD_GROUP_ORDER:
    foods = by_group.get(group, [])
    if not foods:
        continue
    print(f'\n── {group.upper()} ──')
    for key, cfg in foods:
        count = db_paper_count(key)
        status = f'{count:4d} papers' if count is not None else '   (not imported)'
        print(f'  {key:25s}  {cfg["name"]:25s}  {status}')
        if count:
            total_imported += count

existing = len(glob.glob(os.path.join(DATA_DIR, 'food_papers_*.db')))
print(f'\n{existing} food databases on disk, {total_imported:,} papers total')


── VEGETABLES ──
  broccoli                   Broccoli                     18 papers
  carrot                     Carrot                       23 papers
  sweet_potato               Sweet Potato                 35 papers
  spinach                    Spinach                       1 papers
  pea                        Peas                         68 papers
  avocado                    Avocado                      13 papers
  tomato                     Tomato                       37 papers
  zucchini                   Zucchini                      2 papers
  cauliflower                Cauliflower                  12 papers
  beetroot                   Beetroot                      0 papers
  parsnip                    Parsnip                       0 papers
  butternut_squash           Butternut Squash              1 papers
  green_bean                 Green Bean                   44 papers
  kale                       Kale                         32 papers
  bell_pepper                B

## Step 3 — Configure & import

Edit `FOODS_TO_IMPORT` to select which foods to fetch.  
Use `list(PREDEFINED_FOODS.keys())` to import all foods.

`MAX_PAPERS` controls how many papers to fetch per food from OpenAlex.  
`MIN_CITATIONS` filters out papers with fewer citations (0 = include everything).

In [3]:
# ── configure here ────────────────────────────────────────────────────────────

# Import a hand-picked selection:
# FOODS_TO_IMPORT = [
#     'broccoli',
#     'carrot',
#     'sweet_potato',
#     'apple',
#     'banana',
#     'salmon',
#     'egg',
#     'chicken',
#     'oats',
#     'lentil',
#     'yogurt',
#     'breast_milk',
#     'avocado',
#     'peanut',
# ]

# Or import everything:
FOODS_TO_IMPORT = list(PREDEFINED_FOODS.keys())

MAX_PAPERS    = 300   # papers per food
MIN_CITATIONS = 0     # set higher (e.g. 5) to skip low-impact papers
SKIP_EXISTING = False  # skip foods that already have papers (empty DBs are re-imported)

# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DATA_DIR, exist_ok=True)

to_run = []
for key in FOODS_TO_IMPORT:
    if key not in PREDEFINED_FOODS:
        print(f'  [warn] unknown food key: {key} — skipping')
        continue
    existing_count = db_paper_count(key)
    if SKIP_EXISTING and existing_count:  # only skip if count > 0
        print(f'  [skip] {key} — already has {existing_count} papers')
        continue
    to_run.append(key)

print(f'\nWill import {len(to_run)} food(s): {", ".join(to_run)}')


Will import 98 food(s): broccoli, carrot, sweet_potato, spinach, pea, avocado, tomato, zucchini, cauliflower, beetroot, parsnip, butternut_squash, green_bean, kale, bell_pepper, cucumber, leek, apple, banana, mango, strawberry, blueberry, pear, orange, grape, watermelon, kiwi, papaya, peach, plum, raspberry, apricot, chicken, beef, salmon, sardine, egg, lentil, tuna, tofu, cod, tempeh, cows_milk, yogurt, cheese, breast_milk, butter, kefir, oats, rice, wheat, quinoa, corn, barley, millet, bread, buckwheat, chickpea, kidney_bean, peanut, soy, almond, black_bean, cashew, walnut, sunflower_seeds, pumpkin_seeds, tahini, olive_oil, coconut_oil, flaxseed, chia_seed, ghee, hemp_seed, probiotic_food, prebiotic_food, dark_chocolate, herbs_spices, turmeric, ginger, fortified_cereal, turkey, lamb, pork, duck, venison, rabbit, dates, raisins, honey, maple_syrup, rice_cake, fruit_puree, water, formula_milk, fruit_juice, coconut_water, herbal_tea


In [4]:
results = {}
for i, key in enumerate(to_run, 1):
    cfg = PREDEFINED_FOODS[key]
    print(f'\n[{i}/{len(to_run)}] {cfg["name"]} ({key})')
    print(f'  Query: "{cfg["query"]}')
    try:
        import_food(key, max_results=MAX_PAPERS, min_citations=MIN_CITATIONS)
        count = db_paper_count(key) or 0
        results[key] = count
        print(f'  → {count} papers in database')
    except Exception as e:
        print(f'  [error] {e}')
        results[key] = 0

print(f'\nDone. {sum(results.values()):,} papers imported across {len(results)} foods.')


[1/98] Broccoli (broccoli)
  Query: "broccoli infant child nutrition
[import] Food:  Broccoli
[import] Query: broccoli infant child nutrition
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\food_papers_broccoli.db
[import] Existing papers: 18
[import] Total in OpenAlex: 1,039
  Fetched 400 papers total.   
[import] Done: 293 inserted, 7 skipped
  → 311 papers in database

[2/98] Carrot (carrot)
  Query: "carrot infant child nutrition
[import] Food:  Carrot
[import] Query: carrot infant child nutrition
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\food_papers_carrot.db
[import] Existing papers: 23
[import] Total in OpenAlex: 2,458
  Fetched 400 papers total.   
[import] Done: 297 inserted, 3 skipped
  → 320 papers in database

[3/98] Sweet Potato (sweet_potato)
  Query: "sweet potato yam infant child nutrition
[import] Food:  Sweet Potato
[import] Query: sweet potato yam infant child nutrition
[import] DB:    C:\Users\TomHi\Documents\GitHub\map

## Step 4 — Build frontend JSON

Reads all `food_papers_*.db` files and writes:
- `frontend/public/food_universe.json` — the food map index
- `frontend/public/food_data/<food>/nodes.json`
- `frontend/public/food_data/<food>/edges.json`

Refresh the browser after this runs.

In [5]:
os.makedirs(OUT_DIR, exist_ok=True)
build_food_universe(DATA_DIR, OUT_DIR)
print('\nFrontend JSON built. Refresh http://localhost:5173 to see the Food Map.')

[build] Found 98 food databases
[build] Processing almond...
  [ok] almond: 320 papers, 172 citations
[build] Processing apple...
  [ok] apple: 313 papers, 70 citations
[build] Processing apricot...
  [ok] apricot: 300 papers, 22 citations
[build] Processing avocado...
  [ok] avocado: 307 papers, 79 citations
[build] Processing banana...
  [ok] banana: 314 papers, 98 citations
[build] Processing barley...
  [ok] barley: 302 papers, 116 citations
[build] Processing beef...
  [ok] beef: 313 papers, 72 citations
[build] Processing beetroot...
  [ok] beetroot: 35 papers, 1 citations
[build] Processing bell_pepper...
  [ok] bell_pepper: 21 papers, 1 citations
[build] Processing black_bean...
  [ok] black_bean: 344 papers, 117 citations
[build] Processing blueberry...
  [ok] blueberry: 305 papers, 14 citations
[build] Processing bread...
  [ok] bread: 304 papers, 102 citations
[build] Processing breast_milk...
  [ok] breast_milk: 586 papers, 578 citations
[build] Processing broccoli...
  [ok

## Step 4b — Download food images

Downloads a real food photo from Wikipedia for each food node and saves it to
`frontend/public/images/foods/<icon_category>.jpg`.

Images are used as node thumbnails in the Food Map. Missing images fall back to
a plain coloured circle, so this step is optional but recommended.

- `FORCE_DOWNLOAD = False` — skip foods that already have an image
- `FORCE_DOWNLOAD = True` — re-download everything (useful after adding new foods)

In [9]:
import sys, os, time
from io import BytesIO

import requests
from PIL import Image

sys.path.insert(0, BACKEND_DIR)
from download_food_images import (
    icon_to_wikipedia_title,
    fetch_wikipedia_thumbnail,
    download_and_save,
    GROUP_IMAGES,
)

IMAGES_DIR = os.path.normpath(os.path.join(OUT_DIR, "images", "foods"))
os.makedirs(IMAGES_DIR, exist_ok=True)

# ── configure here ────────────────────────────────────────────────────────────
FORCE_DOWNLOAD    = False   # True = re-download even if image already exists
INCLUDE_GROUPS    = True    # also download group-anchor hexagon images
DOWNLOAD_DELAY    = 5     # seconds between requests (avoids Wikipedia 429s)

# Which foods to download images for — default: all predefined foods
# FOODS_FOR_IMAGES = ["broccoli", "salmon", "egg"]
FOODS_FOR_IMAGES  = list(PREDEFINED_FOODS.keys())
# ─────────────────────────────────────────────────────────────────────────────

# Build list of (icon_category, label) pairs to download
seen = set()
targets = []
for key in FOODS_FOR_IMAGES:
    cfg = PREDEFINED_FOODS.get(key)
    if not cfg:
        print(f"  [warn] unknown food key: {key} — skipping")
        continue
    ic = cfg["icon_category"]
    if ic not in seen:
        targets.append((ic, cfg["name"]))
        seen.add(ic)

if INCLUDE_GROUPS:
    for ic, wiki_title in GROUP_IMAGES.items():
        if ic not in seen:
            targets.append((ic, ic))
            seen.add(ic)

already  = sum(1 for ic, _ in targets if os.path.exists(os.path.join(IMAGES_DIR, f"{ic}.jpg")))
to_fetch = [(ic, lbl) for ic, lbl in targets
            if FORCE_DOWNLOAD or not os.path.exists(os.path.join(IMAGES_DIR, f"{ic}.jpg"))]

print(f"Images dir : {IMAGES_DIR}")
print(f"Total foods: {len(targets)}  |  already on disk: {already}  |  to download: {len(to_fetch)}")
if not to_fetch:
    print("Nothing to download — set FORCE_DOWNLOAD=True to re-download all.")

Images dir : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\frontend\public\images\foods
Total foods: 109  |  already on disk: 41  |  to download: 68


In [10]:
ok_count = fail_count = skip_count = 0

for i, (ic, label) in enumerate(to_fetch, 1):
    out_path = os.path.join(IMAGES_DIR, f"{ic}.jpg")
    print(f"[{i}/{len(to_fetch)}] {label} ({ic})", end=" ... ", flush=True)

    wiki_title = icon_to_wikipedia_title(ic)
    thumb_url  = fetch_wikipedia_thumbnail(wiki_title)
    if not thumb_url:
        print(f"no Wikipedia image for '{wiki_title}'")
        fail_count += 1
    elif download_and_save(thumb_url, out_path):
        print("saved")
        ok_count += 1
    else:
        print("download failed")
        fail_count += 1

    time.sleep(DOWNLOAD_DELAY)

total_on_disk = sum(1 for ic, _ in targets if os.path.exists(os.path.join(IMAGES_DIR, f"{ic}.jpg")))
print(f"\nDone: {ok_count} downloaded, {fail_count} failed.")
print(f"Total images on disk: {total_on_disk}/{len(targets)}")
if fail_count:
    print("Re-run this cell to retry failed downloads (Wikipedia rate-limits lift quickly).")

[1/68] Cucumber (cucumber) ... saved
[2/68] Apple (apple) ... saved
[3/68] Banana (banana) ... saved
[4/68] Mango (mango) ... saved
[5/68] Strawberry (strawberry) ... saved
[6/68] Blueberry (blueberry) ... saved
[7/68] Pear (pear) ... saved
[8/68] Orange (orange-fruit) ... saved
[9/68] Grape (grape) ... saved
[10/68] Watermelon (watermelon) ... saved
[11/68] Kiwi (kiwi) ... no Wikipedia image for 'Kiwi'
[12/68] Raspberry (raspberry) ... saved
[13/68] Beef (beef) ... saved
[14/68] Salmon (salmon-fish) ... saved
[15/68] Egg (egg-food) ... saved
[16/68] Breast Milk (breast-milk-f) ... saved
[17/68] Kefir (kefir) ... saved
[18/68] Oats (oats-food) ... saved
[19/68] Rice (rice-food) ... saved
[20/68] Wheat (wheat-food) ...     [warn] Wikipedia API error for 'Wheat': 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&titles=Wheat&prop=pageimages&format=json&pithumbsize=200&redirects=1
no Wikipedia image for 'Wheat'
[21/68] Quinoa (quinoa) ... saved
[

## Step 5 — Inspect what was imported

In [6]:
dbs = sorted(glob.glob(os.path.join(DATA_DIR, 'food_papers_*.db')))
if not dbs:
    print('No food databases found. Run Step 3 first.')
else:
    total = 0
    print(f'{"Food":25s} {"Group":12s} {"Papers":>8s} {"Max Citations":>14s} {"Year range":>12s}')
    print('-' * 80)
    for db_path in dbs:
        key = os.path.basename(db_path).replace('food_papers_', '').replace('.db', '')
        cfg = PREDEFINED_FOODS.get(key, {})
        conn = sqlite3.connect(db_path)
        try:
            row = conn.execute(
                'SELECT COUNT(*) as n, MAX(cited_by_count) as max_c, '
                'MIN(year) as yr0, MAX(year) as yr1 FROM papers'
            ).fetchone()
            n, max_c, yr0, yr1 = row
            yr_range = f'{yr0}–{yr1}' if yr0 and yr1 else 'unknown'
            print(f'{cfg.get("name", key):25s} {cfg.get("group", ""):12s} {n:>8,d} {(max_c or 0):>14,d} {yr_range:>12s}')
            total += n
        except Exception as e:
            print(f'{key}: error — {e}')
        finally:
            conn.close()
    print('-' * 80)
    print(f'{"TOTAL":25s} {"":12s} {total:>8,d}')

Food                      Group          Papers  Max Citations   Year range
--------------------------------------------------------------------------------
Almond                    legumes           320          2,567    1921–2025
Apple                     fruits            313          1,507    1931–2025
Apricot                   fruits            300          2,567    1910–2026
Avocado                   vegetables        307            661    1933–2025
Banana                    fruits            314          2,567    1931–2026
Barley                    grains            302          1,683    1914–2024
Beef                      proteins          313          1,518    1921–2024
Beetroot                  vegetables         35             51    1953–2026
Bell Pepper               vegetables         21            377    2002–2025
Black Bean                legumes           344          2,567    1930–2026
Blueberry                 fruits            305            377    1910–2026
Bread  

## Step 5 — AI Recommendation Summaries (optional)

Uses a **local Ollama/Mistral** model to generate a concise recommendation summary
for each paper. Summaries are stored back in each food database and surfaced in the
frontend when a paper node is selected.

**Prerequisites:**
- Ollama running: `ollama serve`
- Mistral pulled: `ollama pull mistral`

This step is **separate from the import** — run it after Step 3 once you have papers in the DBs.

**After Step 5 run Step 4 again** to rebuild the frontend JSON with enriched metadata.

In [ ]:
from process_food_ai import process_db, get_client

# ── configure here ────────────────────────────────────────────────────────────
AI_MODEL    = "mistral"       # change to e.g. "llama3" if you prefer
BATCH_SIZE  = 10              # papers per commit (lower = less memory)
FORCE_REDO  = False           # set True to re-process already-enriched papers

# Which food DBs to process — default: all that exist
# FOODS_TO_PROCESS = ["broccoli", "salmon", "egg"]
FOODS_TO_PROCESS = [f.replace("food_papers_", "").replace(".db", "")
                    for f in glob.glob(os.path.join(DATA_DIR, "food_papers_*.db"))]
# ─────────────────────────────────────────────────────────────────────────────

client, model = get_client(model=AI_MODEL)

# Quick connectivity check
try:
    client.models.list()
    print(f"Ollama connected. Model: {model}")
except Exception as e:
    print(f"[error] Cannot reach Ollama: {e}")
    print("Run: ollama serve && ollama pull mistral")
    FOODS_TO_PROCESS = []

print(f"{len(FOODS_TO_PROCESS)} food DB(s) queued for enrichment.")

In [ ]:
total_done = 0
for food_key in FOODS_TO_PROCESS:
    db_path = os.path.join(DATA_DIR, f"food_papers_{food_key}.db")
    if not os.path.exists(db_path):
        print(f"  [skip] {food_key} — no DB found")
        continue
    total_done += process_db(db_path, client, model, BATCH_SIZE, FORCE_REDO)

print(f"
Done. {total_done} papers enriched.")

## Step 6 — Cluster Similar Recommendations (optional)

Groups papers with similar recommendation summaries using TF-IDF + KMeans.
Clusters are stored as  integers in each food DB.

Requires: `pip install scikit-learn`

After clustering you can inspect which papers share the same theme
(e.g. all papers about age of introduction vs. all about allergy prevention).

In [ ]:
from process_food_ai import cluster_summaries

# Number of clusters per food (None = auto-pick based on paper count)
N_CLUSTERS = None

for food_key in FOODS_TO_PROCESS:
    db_path = os.path.join(DATA_DIR, f"food_papers_{food_key}.db")
    if os.path.exists(db_path):
        cluster_summaries(db_path, n_clusters=N_CLUSTERS)

In [ ]:
# Inspect enrichment coverage for all food DBs
import sqlite3, os, glob

print(f"{'Food':<25s} {'Papers':<10s} {'Enriched':<10s} {'Coverage':<10s}")
print("-" * 60)
for db_path in sorted(glob.glob(os.path.join(DATA_DIR, "food_papers_*.db"))):
    key = os.path.basename(db_path).replace("food_papers_", "").replace(".db", "")
    conn = sqlite3.connect(db_path)
    try:
        total = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        has_cols = any(r[1] == "recommendation_summary" for r in conn.execute("PRAGMA table_info(papers)"))
        enriched = conn.execute("SELECT COUNT(*) FROM papers WHERE recommendation_summary IS NOT NULL").fetchone()[0] if has_cols else 0
        pct = f"{100*enriched//total}%" if total else "—"
        print(f"{key:<25s} {total:<10d} {enriched:<10d} {pct:<10s}")
    finally:
        conn.close()